# 02 – Ingest FX rates to bronze (Frankfurter API)
Incrementally loads daily BRL -> EUR / CZK / USD rates (ECB data) into `<catalog>.bronze.frankfurter_rates`.

- **Incremental:** only dates after the latest date in bronze are requested (watermark).
- **Idempotent:** MERGE on `(rate_date, base_currency, quote_currency)` - re-runs never duplicate rows.
- **Resilient:** timeouts and retries with backoff on transient API errors (`src/ingestion/api_client.py`).
- **Parameters:** `catalog`, `batch_id`. Period and currencies live in `src/ingestion/config.yml` (`fx` section).

In [0]:
import logging
import sys
import uuid
from pathlib import Path

# Make repo root importable (notebook lives in /notebooks)
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Show INFO logs from our code (src.*), keep other libraries quiet
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)-7s %(name)s: %(message)s", force=True)
logging.getLogger("src").setLevel(logging.INFO)

from src.ingestion.autoloader import load_config
from src.ingestion.frankfurter import ingest_fx

dbutils.widgets.text("catalog", "olist_dev", "Target catalog")
dbutils.widgets.text("batch_id", "", "Batch id (job run id)")

CATALOG = dbutils.widgets.get("catalog")
BATCH_ID = dbutils.widgets.get("batch_id") or f"manual-{uuid.uuid4().hex[:8]}"
cfg = load_config(CATALOG)

In [0]:
import pandas as pd

# Errors are not caught here on purpose: a failed API call must fail the job run
summary = ingest_fx(spark, cfg, CATALOG, BATCH_ID)

# pandas handles None values in a one-row summary (Spark cannot infer a type from None)
display(pd.DataFrame([summary]))

In [0]:
# Quick look at the latest rates
display(spark.sql(f"""
    SELECT rate_date, base_currency, quote_currency, rate, _batch_id
    FROM {summary['table']}
    ORDER BY rate_date DESC, quote_currency
    LIMIT 9
"""))